# BPZ Lite: determining zero point shifts and braod types

**Authors:** Sam Schmidt

**Last Successfully Run:** Jun 05, 2026

This notebook will go through a simple example of running a new "pre-estimation" stage that does two things:
- Compute zero point offsets to the photometric bands
- computes "broad types" for the galaxy sample that can be used to train a custom prior in the inform stage

**Note:** The zero point shift coded up here is different from the one in the original BPZ code, this simply finds the mean offset between the best-fit redshift and type fluxes and the measured fluxes, i.e. "shifting" the mean colors to line up with the best fit templates.  Note that the template, and thus flux, of the "best fit" object can change once a zero point shift has been applied, particularly if they are large shifts.  So, this stage may need to be run iteratively to get the "final" zero point shifts.

**Be wary of large shifts!** Zero point shifts should be small, due to slight miscalibrations of photometry, differences between aperture and total mags, etc... If the code predicts very large zero point offsets of more than ~0.1-0.2 mags, then something else could be wrong.  Do not blindly apply shifts that are output by the code!

**For best results, this should be run with spectroscopic data with secure redshifts and with the `only_type` parameter set to `True`**

For the broad types determination, what the code is doing is finding the best-fit SED for each galaxy.  It then looks at `nt_array` to check how the SED list used by BPZ is being mapped to broad type.  For example, the default `nt_array` for the default CWWSB template set is `[1, 2, 5]`, which means that there are 8 total templates, 1 Elliptical (El_B2004a), 2 Spiral (Sbc_B2004a and Scd_B2004a), and 5 Irregular/Starburst SEDs (Im_B2004, SB3_B2004a, SB2_B2004a, ssp_25Myr_z008, and ssp_5Myr_z008).  It will take the best-fit SEDs and return an integer corresponding to which of the "broad types" that corresponds to, in this case 0, 1, or 2 for El, Sp, or Im/SB.  
**This should be run with `only_type` set to `True`**: if you attempt to run without true redshifts, BPZ will essentially run a prior-free likelihood computation and assign the best type to the peak likelihood.  Given the many degeneracies tha occur in photo-z fitting, this can lead to wildly different type fits, and thus should be avoided.

The list of zero point offsets and broad types are output as an hdf5 file.  They will be stored in separate hdf5 groups, the zero point offsets  as `offsets/zp_offsets` and the broad types as `types/broad_type`

A final warning: most spectroscopic datasets are very systematically incomplete, and thus trying to run a custom prior can be fraught with problems.  I do not expect broad_types and custom priors to be computed very often.  
### **Using the default HDFN prior rather than computing your own prior is highly recommended in almost every case**


## Example: add some offsets to data and test zero-point correction

We will run a quick demo where we read in the same example data used in notebooks 02 and 03 for the main BPZ tests, and we'll add offsets to the g and z bands, then run our zero-point correction code to see how well it does at finding those offsets.

The new stage is located in `src/rail/estimation/algos/bpz_preprocess.py` and the stage name is `BPZlitePreEstimator` (since it needs to be run before estimation):

In [ ]:
import os
import qp
import tables_io
import pickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import desc_bpz
from rail.core.data import TableHandle, ModelHandle
from rail.core.stage import RailStage
from rail.utils.path_utils import RAILDIR
from rail.estimation.algos.bpz_lite import BPZliteInformer, BPZliteEstimator

from rail.estimation.algos.bpz_preprocess import BPZlitePreEstimator

In [ ]:
bands = ["u", "g", "r", "i", "z", "y"]
lsst_bands = []
lsst_errs = []
lsst_filts = []
for band in bands:
    lsst_bands.append(f"mag_{band}_lsst")
    lsst_errs.append(f"mag_err_{band}_lsst")
    lsst_filts.append(f"DC2LSST_{band}")
print(lsst_bands)
print(lsst_filts)

First, let's grab the training and test data files and add some offsets:

In [ ]:
trainFile = os.path.join(RAILDIR, 'rail/examples_data/testdata/test_dc2_training_9816.hdf5')
testFile = os.path.join(RAILDIR, 'rail/examples_data/testdata/test_dc2_validation_9816.hdf5')
training_data = tables_io.read(trainFile)
test_data = tables_io.read(testFile)

training_data['photometry']['mag_g_lsst'] -= 0.15
training_data['photometry']['mag_z_lsst'] += 0.2

test_data['photometry']['mag_g_lsst'] -= 0.15
test_data['photometry']['mag_z_lsst'] += 0.2


Let's run BPZ on this shift file with no zero point offsets applied, results should look sub-optimal due to the g- and z-band shifts.

Now, let's set up a dictionary of configuration parameters and set up to run the estimator.

In [ ]:
hdfnfile = os.path.join(RAILDIR, "rail/examples_data/estimation_data/data/CWW_HDFN_prior.pkl")
default_dict = dict(hdf5_groupname="photometry", output="bpz_results_defaultprior.hdf5",
                    bands=lsst_bands, err_bands=lsst_errs, filter_list=lsst_filts,
                    prior_band="mag_i_lsst", no_prior=False)
run_default = BPZliteEstimator.make_stage(name="bpz_def_prior", model=hdfnfile, **default_dict)

In [ ]:
noshift_res = run_default.estimate(test_data)

In [ ]:
zmode = noshift_res().mode(grid=np.linspace(0,3,301)).flatten()

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(test_data['photometry']['redshift'], zmode, s=1, c='k')
plt.plot([0,3],[0,3], 'r--')
plt.xlabel("true redshift", size=12)
plt.ylabel("z mode no zp correction", size=12)

As expected, we see some extra bias ans horizontal features, as redshifts fits are biased by our zero-point shifts.

Now let's run the `BPZlitePreEstimator` stage to determine our zero point offsets:

In [ ]:
zpt_tweak = BPZlitePreEstimator.make_stage(name="first pass", only_type=True, 
                                           output="bpz_offsets_firstpass.hdf5")

In [ ]:
%%time
results = zpt_tweak.estimate(training_data)

Let's see what offsets we came up with:

In [ ]:
results().keys()

In [ ]:
zpoff = np.array(results()['offsets']['zp_offsets'])
print(zpoff)

We see an offset of -0.121 for the g band and +0.183 for the z band, similar to our applied offsets of -.15 and +0.20.  In addition, there are small 0.01-0.05 offsets for the other bands.
    
We are fitting data from the DC2 simulation with CWWSB templates.  The mismatch in SEDs between the "true" data and our SED models can lead to zero point offsets even with perfect photometry, as BPZ sees the difference in color as being due to bad photometry rather than mismatched SED models.  This is the danger inherent in zero point tweaks, it can confuse observational photometry effects with inherent differences between SED models, it is a zeroth order correction to a model misspecification with other causes.  But, it can often improve photo-z results, at least on the training data, so it does have its uses.

Note that we can re-run but now include an initial set of zero-point tweaks applied via the `zp_offsets` parameter. This will likely lead to better broad-type fit, and potential small updates to the zero points, this second pass will output the *total* zp_offset from the initial data, i.e. it will include the offsets from the initial step in the final offsets.

In [ ]:
zpt_type = BPZlitePreEstimator.make_stage(name="second pass", only_type=True,
                                          zp_offsets=zpoff,
                                          output="bpz_offsets_secondpass.hdf5")

In [ ]:
secpass_res = zpt_type.estimate(training_data)

Let's see what our final offsets are:

In [ ]:
zpoff_final = np.array(secpass_res()['offsets']['zp_offsets'])
print(zpoff_final)

Looks good!  It wants a rather large correction to the u-band and y-band zeropoints (not uncommon to have the bluest and reddest bands needing some "correction" due to differences in SEDs), but we are now closer to our artificial offsets for g and z bands.

We can also look at the broad_types that we computed:

In [ ]:
broad_types = secpass_res()['types']['broad_type']
print(broad_types[:10])

## Training a new prior (with zero point offsets)

Let's compute a new prior with the zpt offsets, you can see notebook 02 for more detail on this if needed.  We will compute a new prior that will be written out as `new_hdfn_prior.pkl`.

We will use the broad_types and zero point offsets from our file `bpz_offsets_secondpass.hdf5`.  There is a parameter `override_file_offsets`, if this is set to `True` you can override the offsets in the file and instead input offsets directly via a `zp_offsets` parameter.  We will **not** do that here.

In [ ]:
train_dict = dict(hdf5_groupname="photometry", model="new_prior.pkl",
                 type_file="./bpz_offsets_secondpass.hdf5",
                 override_file_offsets=False,
                 nt_array=[1,2,5], output_hdfn=False)
run_bpz_train = BPZliteInformer.make_stage(name="bpz_new_prior", **train_dict)

In [ ]:
%%time
run_bpz_train.inform(training_data)

We can compare our new prior to the default HDFN prior, similar to done in notebook 02.  First, the parameters of the HDFN model:

In [ ]:
from desc_bpz.prior_from_dict import prior_function


hdfnfile = os.path.join(RAILDIR, "rail/examples_data/estimation_data/data/CWW_HDFN_prior.pkl")

with open(hdfnfile, "rb") as f:
    xhdfnmodel = pickle.load(f)
hdfnmodel = xhdfnmodel['priormodel']
xhdfnmodel

Now the parameters of our new model:

In [ ]:
with open("./new_prior.pkl", "rb") as f:
    xnewmodel = pickle.load(f)
newmodel = xnewmodel['priormodel']
xnewmodel

We can plot up the prior at a couple of magnitudes to do a more visual comparison:

In [ ]:
zgrid=np.linspace(0,3,301)
defprior20 = prior_function(zgrid, 20., hdfnmodel, 8)
defprior23 = prior_function(zgrid, 23., hdfnmodel, 8)
defprior25 = prior_function(zgrid, 25., hdfnmodel, 8)
newprior23 = prior_function(zgrid, 23., newmodel, 8)
newprior25 = prior_function(zgrid, 25., newmodel, 8)
newprior20 = prior_function(zgrid, 20., newmodel, 8)

In [ ]:
seddict = {'El': 0, 'Sp': 1, 'Irr/SB': 7}
multiplier = [1.0, 2.0, 5.0]
sedcol = ['r', 'm', 'b']
fig, (axs, axs2, axs3) = plt.subplots(3, 1, figsize=(7,10))
for sed, col, multi in zip(seddict, sedcol, multiplier):
    axs.plot(zgrid, defprior20[:,seddict[sed]]*multi, color=col, lw=2,ls='--', label=f"hdfn prior {sed}")
    axs.plot(zgrid, newprior20[:,seddict[sed]]*multi, color=col, ls='-', label=f"new prior {sed}")
    axs.set_title("priors for mag=20.0")
    axs2.plot(zgrid, defprior23[:,seddict[sed]]*multi, color=col, lw=2,ls='--', label=f"hdfn prior {sed}")
    axs2.plot(zgrid, newprior23[:,seddict[sed]]*multi, color=col, ls='-', label=f"new prior {sed}")
    axs2.set_title("priors for mag=23.0")
    axs3.plot(zgrid, defprior25[:,seddict[sed]]*multi, color=col, lw=2,ls='--', label=f"hdfn prior {sed}")
    axs3.plot(zgrid, newprior25[:,seddict[sed]]*multi, color=col, ls='-', label=f"new prior {sed}")
    axs3.set_xlabel("redshift")
    axs3.set_title("priors for mag=25.0")
    axs3.set_ylabel("prior_probability")
    axs.set_ylabel("prior probability")
axs.legend(loc="upper right", fontsize=10)

at mag=20 our new prior predicts almost no Im/SB galaxies and more El and Sp, but otherwise the priors are not extremely different, which is probably a good thing.

Finally, let's compute updated photo-z's with our new prior and new offsets and compare to the old ones:

In [ ]:
rerun_dict = dict(hdf5_groupname="photometry", output="bpz_results_newprior.hdf5",
                  bands=lsst_bands, err_bands=lsst_errs, filter_list=lsst_filts,
                  prior_band="mag_i_lsst", no_prior=False,
                  override_file_offsets=False,
                  )
rerun_bpz = BPZliteEstimator.make_stage(name="bpz_def_prior", model="./new_prior.pkl", **rerun_dict)

In [ ]:
%%time
final_res = rerun_bpz.estimate(test_data)

Let's compare to our earlier, non-shifted and default prior results:

In [ ]:
fin_zmode = final_res().mode(grid=np.linspace(0,3,301)).flatten()
sz = test_data['photometry']['redshift']

In [ ]:
def bias_sigiqr_outlier(sz, zb):
    ez = (zb - sz) / (1. + sz)
    bias = np.median(ez)
    x75, x25 = np.percentile(ez, [75.0, 25.0])
    iqr = x75 - x25
    sigma_iqr = iqr / 1.349
    outmask = np.fabs(ez) > 0.15
    fout = np.sum(outmask)/len(ez)
    return bias, sigma_iqr, fout

bias, sigma, fout = bias_sigiqr_outlier(sz, zmode)
fin_bias, fin_sigma, fin_fout = bias_sigiqr_outlier(sz, fin_zmode)

print(f"bias with no shift: {bias}\nsigma no shift: {sigma}\nbias wshift/offsets: {fin_bias}\nsigma wshift/offsets: {fin_sigma}")
print(f"outfrac no shift: {fout}\noutfrac wshift/offset: {fin_fout}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs[0].scatter(sz, zmode, s=1, c='k')
axs[0].plot([0,3],[0,3], 'r--')
axs[1].scatter(sz, fin_zmode, s=1, c='k')
axs[1].plot([0,3],[0,3], 'r--')

So, a reduction in the scatter, a slight increase in bias, and a reduction of the outlier rate, our prior and zero point tweaks helped, though we are still limited by our use of only 8 slightly mismatched SEDs that do not fully represent the underlying data. 

# Applying zero point shifts directly

Finally, if you want to just apply zero point shifts directly that were determined from some other method, or re-using from other code, you do not need to read via the file, you can override the zero points from the model and apply directly by setting
`override_file_offsets=True` and then supplying an array of offsets (that is the same length as the number of filters/bands) with the `zp_offsets` parameter:

In [ ]:
forced_offsets = np.array([-0.0713074 , -0.1471,  0.0024,  0.0,  0.1883, -0.0486])
zp_dict = dict(hdf5_groupname="photometry", output="bpz_results_newprior.hdf5",
               bands=lsst_bands, err_bands=lsst_errs, filter_list=lsst_filts,
               prior_band="mag_i_lsst", no_prior=False,
               override_file_offsets=True,
               zp_offsets=forced_offsets)
forced_bpz = BPZliteEstimator.make_stage(name="bpz_forced_offsets", model="./new_prior.pkl", **zp_dict)

In [ ]:
forced_res = forced_bpz.estimate(test_data)